# BHW1: Image Classification
Готовый ноутбук для работы локально и в Colab

In [ ]:
# ============== SETUP ==============
import os

# Проверяем, в Colab ли мы
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files
    print('Загрузи kaggle.json:')
    files.upload()
    
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    
    !kaggle competitions download -c bhw-1-dl-2025-2026
    !unzip -q bhw-1-dl-2025-2026.zip -d dataset
    
    DATA_DIR = '/content/dataset/bhw1'
else:
    DATA_DIR = '/Users/amirhamidullin/Downloads/bhw1'

# Универсальные пути
TRAIN_DIR = os.path.join(DATA_DIR, 'trainval')
TEST_DIR = os.path.join(DATA_DIR, 'test')
LABELS_CSV = os.path.join(DATA_DIR, 'labels.csv')
SUBMISSION_CSV = os.path.join(DATA_DIR, 'sample_submission.csv')

print(f'IN_COLAB: {IN_COLAB}')
print(f'DATA_DIR: {DATA_DIR}')

In [ ]:
# ============== IMPORTS ==============
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Device: {device}')

In [ ]:
# ============== DATASET ==============
class CustomImageDataset(Dataset):
    def __init__(self, annotations_df, img_dir, transform=None):
        self.img_labels = annotations_df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # BGR -> RGB
        label = int(self.img_labels.iloc[idx, 1])  # Явно int!
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [ ]:
# ============== TRANSFORMS ==============
train_transform = T.Compose([
    T.ToImage(),
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.ToImage(),
    T.Resize((224, 224)),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# ============== DATA LOADING ==============
data = pd.read_csv(LABELS_CSV)
df_test = pd.read_csv(SUBMISSION_CSV)

df_train, df_valid = train_test_split(
    data, test_size=0.2, random_state=42, stratify=data.Category
)

print(f'Train: {len(df_train)}, Valid: {len(df_valid)}, Test: {len(df_test)}')

# Datasets
dataset_train = CustomImageDataset(df_train, TRAIN_DIR, transform=train_transform)
dataset_val = CustomImageDataset(df_valid, TRAIN_DIR, transform=val_transform)
dataset_test = CustomImageDataset(df_test, TEST_DIR, transform=val_transform)

# DataLoaders (БЕЗ iter!)
BATCH_SIZE = 32
train_loader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')

In [ ]:
# ============== ПРОВЕРКА ДАННЫХ ==============
images, labels = next(iter(train_loader))
print(f'Images shape: {images.shape}')
print(f'Labels dtype: {labels.dtype}')
print(f'Labels sample: {labels[:5]}')

In [ ]:
# ============== MODEL ==============
class ConvNet(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.AdaptiveAvgPool2d(1)  # Универсальный пулинг
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.max_pool2d(x, 2)
        
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Проверка модели
model = ConvNet().to(device)
dummy = torch.randn(1, 3, 224, 224).to(device)
print(f'Output shape: {model(dummy).shape}')

In [ ]:
# ============== TRAINING FUNCTIONS ==============
def training_epoch(model, optimizer, criterion, train_loader, tqdm_desc):
    model.train()
    train_loss, train_accuracy = 0.0, 0.0
    
    for images, labels in tqdm(train_loader, desc=tqdm_desc):
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.shape[0]
        train_accuracy += (logits.argmax(dim=1) == labels).sum().item()
    
    train_loss /= len(train_loader.dataset)
    train_accuracy /= len(train_loader.dataset)
    return train_loss, train_accuracy


@torch.no_grad()
def validation_epoch(model, criterion, val_loader, tqdm_desc):
    model.eval()
    val_loss, val_accuracy = 0.0, 0.0
    
    for images, labels in tqdm(val_loader, desc=tqdm_desc):
        images = images.to(device)
        labels = labels.to(device)
        
        logits = model(images)
        loss = criterion(logits, labels)
        
        val_loss += loss.item() * images.shape[0]
        val_accuracy += (logits.argmax(dim=1) == labels).sum().item()
    
    val_loss /= len(val_loader.dataset)
    val_accuracy /= len(val_loader.dataset)
    return val_loss, val_accuracy

In [ ]:
# ============== PLOT FUNCTION ==============
def plot_losses(train_losses, val_losses, train_accs, val_accs):
    from IPython.display import clear_output
    clear_output(wait=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(train_losses, label='train')
    axes[0].plot(val_losses, label='val')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].legend()
    axes[0].set_title('Loss')
    
    axes[1].plot(train_accs, label='train')
    axes[1].plot(val_accs, label='val')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('accuracy')
    axes[1].legend()
    axes[1].set_title('Accuracy')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ============== TRAIN LOOP ==============
def train(model, optimizer, scheduler, criterion, train_loader, val_loader, num_epochs):
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = training_epoch(
            model, optimizer, criterion, train_loader,
            tqdm_desc=f'Training {epoch}/{num_epochs}'
        )
        val_loss, val_acc = validation_epoch(
            model, criterion, val_loader,
            tqdm_desc=f'Validating {epoch}/{num_epochs}'
        )

        if scheduler is not None:
            scheduler.step()

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        plot_losses(train_losses, val_losses, train_accs, val_accs)
        print(f'Epoch {epoch}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}')

    return train_losses, val_losses, train_accs, val_accs

In [ ]:
# ============== RUN TRAINING ==============
NUM_EPOCHS = 10
LR = 1e-3

model = ConvNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

train_losses, val_losses, train_accs, val_accs = train(
    model, optimizer, scheduler, criterion, train_loader, val_loader, NUM_EPOCHS
)

In [ ]:
# ============== PREDICTIONS ==============
@torch.no_grad()
def predict(model, test_loader):
    model.eval()
    predictions = []
    
    for images, _ in tqdm(test_loader, desc='Predicting'):
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1)
        predictions.extend(preds.cpu().numpy())
    
    return predictions

preds = predict(model, test_loader)
print(f'Predictions: {len(preds)}')

In [ ]:
# ============== SUBMISSION ==============
submission = pd.read_csv(SUBMISSION_CSV)
submission['Category'] = preds
submission.to_csv('submission.csv', index=False)

print('Saved submission.csv')
submission.head()